# AI-Powered Fake News Detection Using Text Classification

**Project 1 — Summer Internship Program in AI & ML, 2026**

**Problem Statement:** Build a machine learning pipeline to classify news articles as
*real* or *fake*, covering preprocessing, feature extraction, model training, and
evaluation, and comparing parametric vs. non-parametric algorithms.

**Note on "from scratch":** the algorithm list specified in the problem statement
(KNN, Logistic Regression, Random Forest, Neural Network) refers to scikit-learn's
implementations of these models. "From scratch" is interpreted here as: the text
cleaning/preprocessing logic is hand-written rather than using a pre-built fake-news
detector or pretrained classifier, while vectorization (TF-IDF/BoW) and the model
implementations themselves use scikit-learn, consistent with the algorithms named
in the problem statement.

## 1. Imports

In [ ]:
# Core
import pandas as pd
import numpy as np
import re

# Splitting & pipeline
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Visualization
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 80)

: 

## 2. Data Loading & Dataset Description

Dataset: Kaggle "Fake and Real News Dataset" (`True.csv` / `Fake.csv`).
Place both files in the same directory as this notebook before running.

In [ ]:
# Load both datasets
true_df = pd.read_csv("True.csv")
fake_df = pd.read_csv("Fake.csv")

# Create labels (1 = real, 0 = fake)
true_df['label'] = 1
fake_df['label'] = 0

# Combine the datasets
data = pd.concat([true_df, fake_df], ignore_index=True)

# Shuffle the rows
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Features and labels
X = data['text']
y = data['label']

In [ ]:
# Dataset description -- size, structure, class balance, sample rows
print("Shape:", data.shape)
print()
print(data.info())
print()
print("Class balance (1 = real, 0 = fake):")
print(data['label'].value_counts())
print()
data.head(3)

## 3. Exploratory Data Analysis (EDA)

Before cleaning, we look at class balance and article length to understand the
raw data and spot potential issues (e.g. length being a trivial giveaway feature).

In [ ]:
# Class balance plot
data['label'].map({1: 'Real', 0: 'Fake'}).value_counts().plot(
    kind='bar', title='Class Balance', ylabel='Article count', rot=0
)
plt.tight_layout()
plt.show()

In [ ]:
# Article length distribution by class
data['text_length'] = data['text'].str.len()

data.groupby('label')['text_length'].describe()

In [ ]:
# Visual comparison of article length: real vs fake
fig, ax = plt.subplots(figsize=(8, 4))
data[data['label'] == 1]['text_length'].plot(kind='hist', bins=50, alpha=0.6, label='Real', ax=ax)
data[data['label'] == 0]['text_length'].plot(kind='hist', bins=50, alpha=0.6, label='Fake', ax=ax)
ax.set_xlabel('Character length')
ax.set_title('Article Length Distribution by Class')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Most frequent raw words per class (before cleaning) -- quick sanity check
from collections import Counter

def top_words(series, n=15):
    words = ' '.join(series).lower().split()
    return Counter(words).most_common(n)

print("Top words -- Real:", top_words(data[data['label'] == 1]['text']))
print()
print("Top words -- Fake:", top_words(data[data['label'] == 0]['text']))

**Observation:** words like `reuters`, `washington`, `read more`, and
`featured image` show up disproportionately as boilerplate/metadata rather than
content -- mostly because real-news articles in this dataset consistently carry a
Reuters dateline while the fake-news articles don't. Left in, these act as label
leakage: a model could learn to spot "Reuters" instead of learning anything about
the actual language of misinformation. They are removed during cleaning below.

## 4. Text Cleaning & Tokenization

In [ ]:
def clean_text(text):
    text = re.sub(r'\W', ' ', text)   # remove punctuation
    text = text.lower()

    # Remove metadata identified as leakage during EDA
    for word in ['reuters', 'washington', 'read more', 'featured image']:
        text = text.replace(word, ' ')

    text = re.sub(r'\s+', ' ', text).strip()  # collapse extra whitespace
    return text

X = X.apply(clean_text)

In [ ]:
def manual_tokenize(text):
    """Simple whitespace tokenizer, applied manually per the Week 1 requirement.
    TfidfVectorizer/CountVectorizer perform their own internal tokenization during
    fit_transform, but this step demonstrates the manual tokenization explicitly."""
    return text.split()

X_tokens = X.apply(manual_tokenize)

# Sanity check
print("Sample cleaned text:", X.iloc[0][:120])
print("Sample tokens:", X_tokens.iloc[0][:15])
print("Average tokens per article:", X_tokens.apply(len).mean().round(1))

## 5. Train/Test Split

In [ ]:
# Train/test split, shuffle & stratify to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print("Train class balance:")
print(y_train.value_counts(normalize=True))
print("Test class balance:")
print(y_test.value_counts(normalize=True))

## 6. Feature Extraction -- Bag-of-Words vs. TF-IDF

The workflow calls for implementing both Bag-of-Words and TF-IDF. Below, both are
quickly compared using Logistic Regression as a fixed reference model, before
committing to TF-IDF (which down-weights very common words and generally performs
at least as well as raw counts) for the full four-model comparison in Section 7.

In [ ]:
bow_pipeline = Pipeline([
    ("bow", CountVectorizer(max_features=5000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000)),
])
bow_pipeline.fit(X_train, y_train)
bow_acc = accuracy_score(y_test, bow_pipeline.predict(X_test))

tfidf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000)),
])
tfidf_pipeline.fit(X_train, y_train)
tfidf_acc = accuracy_score(y_test, tfidf_pipeline.predict(X_test))

print(f"Logistic Regression -- Bag-of-Words accuracy: {bow_acc:.4f}")
print(f"Logistic Regression -- TF-IDF accuracy:        {tfidf_acc:.4f}")

**Decision:** TF-IDF is used for the full model comparison in Section 7, since it
weights discriminative words more heavily than boilerplate high-frequency words,
typically giving equal or better accuracy than raw Bag-of-Words counts.

## 7. Model Building & Evaluation

Each model is wrapped in its own `Pipeline` with a `TfidfVectorizer`, so every model
sees identical features and the comparison is apples-to-apples. Random states are
fixed for reproducibility.

In [ ]:
models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42),
}

results = {}
fitted_pipelines = {}

for name, clf in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
        ("clf", clf),
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }
    fitted_pipelines[name] = pipeline

    print(f"{name} Accuracy: {results[name]['accuracy']:.4f}")
    print(classification_report(y_test, preds))
    print("-" * 60)

### Results summary table

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("accuracy", ascending=False)
results_df.style.format("{:.4f}").background_gradient(cmap="Blues")

## 8. Accuracy Comparison

In [ ]:
results_df["accuracy"].plot(
    kind="bar", title="Model Accuracy Comparison", ylabel="Accuracy", rot=0
)
plt.ylim(0.6, 1)
plt.tight_layout()
plt.show()

## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))

for ax, (name, pipeline) in zip(axes, fitted_pipelines.items()):
    ConfusionMatrixDisplay.from_estimator(
        pipeline, X_test, y_test, ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 10. Discussion -- Parametric vs. Non-Parametric Models

- **Logistic Regression (parametric):** learns a fixed-size weight vector over the
  5000 TF-IDF features, which scales well to high-dimensional sparse text data --
  this is typically the strongest simple baseline for text classification.
- **KNN (non-parametric):** classifies by distance to the nearest training points
  in 5000-dimensional TF-IDF space. High dimensionality makes distances between
  points less meaningful (the "curse of dimensionality"), and sparse TF-IDF vectors
  don't suit KNN's distance metrics well -- this typically shows up as a noticeably
  lower accuracy than the other three models here.
- **Random Forest (ensemble, non-parametric):** handles high-dimensional sparse
  features reasonably well by splitting on individual informative words/features,
  and benefits from averaging across many trees to reduce overfitting.
- **Neural Network / MLP (parametric, deep learning):** can model non-linear
  interactions between TF-IDF features, generally matching or slightly exceeding
  Logistic Regression, at the cost of longer training time and less interpretability.

Fill in the actual numbers from Section 7's results table once the notebook has
been run on the full dataset, and connect them to this discussion in the final report.

## 11. Conclusion (draft notes -- expand in final report)

- Which model performed best and why, referencing Section 10.
- Key preprocessing decision that mattered most (e.g. removing the Reuters/dateline
  leakage identified in EDA).
- Limitations: dataset is time/source-specific (2016-era US political news), so
  generalization to other domains/languages is untested.
- Future scope: word embeddings (Word2Vec/GloVe/transformers), larger vocabulary,
  cross-domain validation, real-time NewsAPI scraping for live testing.

## 12. Environment (for Appendix / reproducibility)

In [ ]:
import sklearn, matplotlib
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)